# Model Distillation
## AIAT 122 – Deep Learning

## Learning objectives
- Train a small **student** model to mimic a larger **teacher** using soft labels.
- Compare teacher vs student accuracy and size.

**Where is this used in real life?** Deploying big models on phones or edge is costly. **We use distillation to get a small student that mimics the teacher** instead of only training a small model from hard labels because the teacher’s soft probabilities carry more information (e.g. “maybe class 2”) than one-hot labels.

**Prerequisites:** Basic TensorFlow/Keras. If TensorFlow import fails, see DOCS/COLAB_SETUP.md.

**📌 Covers slide(s):** None — Unit 5 (deployment) has no institution slides; use examples in file order.


## Short theory
- **Distillation:** Train a small student to match the **soft outputs** (probabilities) of a trained teacher.
- **Soft labels:** Teacher outputs P(class) instead of one-hot; student learns from these (often with temperature T to soften further).
- **Why we use distillation:** A small network can approximate the teacher’s behavior and run faster; soft labels help the student generalize.

## Inputs & Outputs
**Inputs:** TensorFlow, synthetic or small dataset, teacher and student models.  
**Dataset:** Synthetic — random data (no download; used to demonstrate distillation).  
**Outputs:** Trained teacher and student, accuracy of both, and a bar chart comparing them. Run time: under ~5 min (few epochs).


In [ ]:
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
print(f'PyTorch {torch.__version__}')
print('✅ Ready. Demonstrating knowledge distillation in PyTorch.')

### Step 1: Build and train teacher (larger model)

In [ ]:
# Synthetic 3-class data
np.random.seed(42)
X = torch.tensor(np.random.randn(400, 20).astype(np.float32))
y_hard = torch.tensor((X[:, :3].argmax(1).numpy()))  # hard labels 0/1/2

# Teacher — large model, train it first
teacher = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 3))
opt_t = optim.Adam(teacher.parameters(), lr=3e-3)
crit  = nn.CrossEntropyLoss()
for _ in range(50):
    opt_t.zero_grad(); loss = crit(teacher(X), y_hard); loss.backward(); opt_t.step()
with torch.no_grad():
    acc_teacher = (teacher(X).argmax(1) == y_hard).float().mean().item()
print(f'Teacher accuracy: {acc_teacher:.3f}')

### Step 2: Get soft labels and train student (smaller model)

In [ ]:
# Distillation: student learns soft labels from the teacher
# Soft labels carry relative confidence — more informative than one-hot targets.
temperature = 4.0

with torch.no_grad():
    soft_labels = F.softmax(teacher(X) / temperature, dim=1)  # soft targets

# Student — small model
student = nn.Sequential(nn.Linear(20, 16), nn.ReLU(), nn.Linear(16, 3))
opt_s = optim.Adam(student.parameters(), lr=3e-3)

for _ in range(50):
    opt_s.zero_grad()
    logits_s = student(X)
    # KL-divergence loss against teacher soft labels
    soft_loss = F.kl_div(F.log_softmax(logits_s / temperature, dim=1),
                         soft_labels, reduction='batchmean') * (temperature ** 2)
    hard_loss = crit(logits_s, y_hard)
    loss = 0.5 * soft_loss + 0.5 * hard_loss  # blend
    loss.backward(); opt_s.step()

with torch.no_grad():
    acc_student = (student(X).argmax(1) == y_hard).float().mean().item()
print(f'Student accuracy: {acc_student:.3f}')
print(f'Teacher params: {sum(p.numel() for p in teacher.parameters())}  |  '
      f'Student params: {sum(p.numel() for p in student.parameters())}')

In [ ]:
# Compare teacher vs student accuracy
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Teacher (large)', 'Student (distilled)'], [acc_teacher, acc_student],
       color=['#3498db', '#2ecc71'])
ax.set_ylim(0, 1); ax.set_ylabel('Accuracy')
ax.set_title('Knowledge Distillation: teacher → smaller student')
plt.tight_layout(); plt.show()
print('Key insight: the compact student reaches close to teacher accuracy.')

## 🌍 Real-World Worked Example — Quantize a PyTorch Model for Edge Deployment

**Industry context:**
- Apple ships quantized CoreML models in every iPhone (face recognition, Siri)
- Google runs quantized TFLite models on Pixel phones for camera AI
- NVIDIA's Jetson Nano (used in robotics) requires 8-bit quantized models to run in real-time

We demonstrate **dynamic quantization** — shrinking a model and measuring size + speed gains.

In [ ]:
import torch, torch.nn as nn
import numpy as np, time, os

# ── Build and train a model ────────────────────────────────────────────────
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

digits = load_digits()
X = StandardScaler().fit_transform(digits.data.astype(np.float32))
y = digits.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)
Xtr=torch.tensor(X_tr); Ytr=torch.tensor(y_tr,dtype=torch.long)
Xte=torch.tensor(X_te); Yte=torch.tensor(y_te,dtype=torch.long)

class BigModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64,512), nn.ReLU(),
            nn.Linear(512,512), nn.ReLU(),
            nn.Linear(512,256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)

model = BigModel()
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(300):
    loss = nn.CrossEntropyLoss()(model(Xtr), Ytr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = (model(Xte).argmax(1)==Yte).float().mean().item()
print(f"Original model accuracy: {acc*100:.1f}%")

# ── Dynamic Quantization (FP32 → INT8) ────────────────────────────────────
# On Apple Silicon the default quantized engine is unset — select qnnpack
if 'qnnpack' in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = 'qnnpack'
quantized = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

# ── Size comparison ────────────────────────────────────────────────────────
torch.save(model.state_dict(),      '/tmp/model_fp32.pt')
torch.save(quantized.state_dict(),  '/tmp/model_int8.pt')
fp32_size = os.path.getsize('/tmp/model_fp32.pt') / 1024
int8_size = os.path.getsize('/tmp/model_int8.pt') / 1024
print(f"\nModel size:  FP32={fp32_size:.1f} KB   INT8={int8_size:.1f} KB  ({fp32_size/int8_size:.1f}x smaller)")

# ── Speed comparison ───────────────────────────────────────────────────────
N = 5000
start=time.perf_counter()
for _ in range(N):
    with torch.no_grad(): model(Xte)
fp32_ms = (time.perf_counter()-start)/N*1000

start=time.perf_counter()
for _ in range(N):
    with torch.no_grad(): quantized(Xte)
int8_ms = (time.perf_counter()-start)/N*1000

print(f"Latency:     FP32={fp32_ms:.3f} ms  INT8={int8_ms:.3f} ms  ({fp32_ms/int8_ms:.1f}x speedup)")

with torch.no_grad():
    qacc = (quantized(Xte).argmax(1)==Yte).float().mean().item()
print(f"Quantized model accuracy: {qacc*100:.1f}%  (accuracy loss: {(acc-qacc)*100:.2f}%)")
print("\nThis is how Apple ships AI features on iPhone without draining the battery.")

## 🧩 Mini-exercise

**Try it:** Change the temperature (e.g. 2.0 → 4.0) when computing soft labels and retrain the student. Does the student accuracy change?

---

## Summary
**What you did:** Trained a teacher (larger) and a student (smaller) on synthetic data; the student learned from the teacher’s soft outputs and achieved comparable accuracy.

**In real life you'd also:** Use temperature scaling for softer distributions, mix soft and hard labels, and deploy only the student.

**The main idea:** Distillation transfers knowledge from a large teacher to a small student via soft labels.

**Next:** `07_model_optimization_quantization.ipynb` shows quantization to reduce model size and speed.

## 📚 References & Further Reading

**Papers:**
- Han et al. (2015) — [Deep Compression (Pruning + Quantization)](https://arxiv.org/abs/1510.00149)
- Hinton et al. (2015) — [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)
- Frantar et al. (2022) — [GPTQ: Post-Training Quantization for LLMs](https://arxiv.org/abs/2210.17323)

**State-of-the-Art:**
- llama.cpp runs LLaMA-7B on your laptop using 4-bit quantization (from 14GB → 4GB)
- Apple Neural Engine runs quantized models at 10x the speed of FP32 on iPhone